<a href="https://colab.research.google.com/github/hbozkurt17/bist-screener/blob/main/FDS_Pro_v3_2_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ========================================================================================================
# ⚙️ STRATEJİ MİMARI VE GELİŞTİRİCİ : @TTesviyeci (Twitter/X)
# 🤖 KODLAMA VE OPTİMİZASYON        : Gemini 3.1 Pro
# 📅 SÜRÜM                          : FDS Pro v3.2 (Python Screener Edition)
# ========================================================================================================
"""
==========================================================================================================
🚀 FDS PRO v3.2 - YAPAY ZEKA DESTEKLİ FİNANSAL TARAMA VE DEĞERLEME SİSTEMİ
==========================================================================================================

📌 SİSTEMİN AMACI:
FDS Pro, Borsa İstanbul (BIST) ve ABD Borsalarındaki (Wall Street) binlerce hisseyi saniyeler
içinde temel analiz kriterlerine göre tarayan, puanlayan ve filtreleyen profesyonel bir analiz motorudur.

⚙️ NASIL ÇALIŞIR? (Veri Akışı ve Mimari):
1. Canlı Makro Veri: 'tvDatafeed' ile TradingView üzerinden anlık Enflasyon ve 10 Yıllık Tahvil
   Faizi (WACC) çekilir. Sistem ekonomik gerçekliğe otomatik adapte olur.
2. Bilanço Verileri: 'tradingview_screener' API'si ile piyasadaki tüm hisselerin 20+ temel verisi
   (F/K, ROIC, Marjlar, Büyüme, Halka Açıklık vb.) tek bir paket halinde indirilir (Limit: 20.000).
3. Vektörel Puanlama: Pandas kütüphanesi kullanılarak 5 ana grupta (Likidite, Kârlılık, Büyüme,
   Nakit Akışı, Değerleme) hisselere 10 üzerinden 'FDS Skoru' verilir.
4. Guru ve Risk Radarı: Peter Lynch (Ucuz Büyüme), Benjamin Graham (Defansif) ve Piotroski (Kalite)
   kriterleri uygulanır. İflas riski veya Değer İllüzyonu taşıyan şirketler radara yakalanır.

🎯 STRATEJİLER (7 Farklı Filtre):
- Kusursuzlar (Kalite + Ucuzluk)
- Yükselen Yıldızlar (Potansiyel İvme)
- Değer Avı (Derin İskonto)
- Temiz Bilanço (Maksimum Güvenlik)
- Filtresiz (Tüm Piyasa Sıralaması)
- Fethi Hoca Modeli (Kalite + Yüksek Hız)
- Değer Yaratanlar (ROIC > WACC - Gerçek Zenginlik)

📊 ÇIKTI VE RAPORLAMA:
- Terminal ekranında renk kodlu (Yeşil/Sarı/Kırmızı) şık bir özet tablo sunar.
- Tüm sonuçları otomatik olarak Excel'e kaydeder. ABD piyasası taramalarında hisseleri borsalarına
  göre (NASDAQ, NYSE, AMEX, OTC) otomatik olarak ayrı sekmelere (sheet) böler.
==========================================================================================================
"""
print("📦 Gerekli kütüphaneler kuruluyor/güncelleniyor...")
!pip install pandas numpy tqdm openpyxl psutil -q
!pip install git+https://github.com/rongardF/tvdatafeed -q
!pip install tradingview-screener==2.5.0 -q
print("✅ Kurulum tamamlandı.\n")

import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime, time as dt_time
import time
import psutil
from tvDatafeed import Interval, TvDatafeed
from tradingview_screener import Query
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ==================================================================================================
# 🔌 TVDATAFEED BAĞLANTISI (Hata Toleranslı Orijinal Yapı)
# ==================================================================================================
try:
    tv = TvDatafeed()
except Exception as e:
    print(f"TvDatafeed bağlantı hatası: {e}. Tekrar deneniyor...")
    time.sleep(2)
    try:
        tv = TvDatafeed()
    except Exception as e2:
        print(f"TvDatafeed ikinci bağlantı hatası: {e2}. Program sonlandırılıyor.")
        exit()

# --- GÖRÜNÜM VE FORMATLAMA ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

# ==================================================================================================
# 🎛️ KULLANICI ETKİLEŞİM MENÜLERİ (CLI)
# ==================================================================================================

def piyasa_secimi():
    print("\n" + "="*60)
    print("🌍 HEDEF PİYASA SEÇİMİ".center(60))
    print("="*60)
    print("1️⃣  : Borsa İstanbul (BIST) - [Canlı Makro Veri]")
    print("2️⃣  : ABD Borsaları (NASDAQ/NYSE) - [Canlı Makro Veri]\n")

    while True:
        secim = input("👉 Seçiminiz (1-2) [Varsayılan: 1]: ").strip() or "1"

        if secim in ["1", "2"]:
            market_code = "turkey" if secim == "1" else "america"
            market_label = "Borsa İstanbul" if secim == "1" else "Wall Street"

            print(f"\n📡 {market_label} için güncel makro veriler TradingView'dan çekiliyor...")
            try:
                if secim == "1":
                    # Türkiye Canlı Verileri
                    inf_data = tv.get_hist(symbol='TRIRYY', exchange='ECONOMICS', interval=Interval.in_daily, n_bars=1)
                    wacc_data = tv.get_hist(symbol='TR10Y', exchange='TVC', interval=Interval.in_daily, n_bars=1)

                    inflation = float(inf_data['close'].iloc[-1]) if inf_data is not None else 45.0
                    wacc = float(wacc_data['close'].iloc[-1]) if wacc_data is not None else 32.1
                else:
                    # ABD Canlı Verileri
                    inf_data = tv.get_hist(symbol='USIRYY', exchange='ECONOMICS', interval=Interval.in_daily, n_bars=1)
                    wacc_data = tv.get_hist(symbol='US10Y', exchange='TVC', interval=Interval.in_daily, n_bars=1)

                    inflation = float(inf_data['close'].iloc[-1]) if inf_data is not None else 3.0
                    # ABD WACC hesabı: 10 Yıllık Tahvil Faizi + %1.5 Hisse Senedi Risk Primi
                    wacc = float(wacc_data['close'].iloc[-1]) + 1.5 if wacc_data is not None else 5.5

                print(f"✅ Canlı Veri Alındı -> Enflasyon: %{inflation:.2f} | WACC: %{wacc:.2f}")
            except Exception as e:
                # İnternet veya API anlık koparsa sistem çökmesin diye yedek (fallback) değerler
                print(f"⚠️ Canlı veri çekilemedi! Varsayılan değerler kullanılıyor. ({e})")
                inflation = 45.0 if secim == "1" else 3.0
                wacc = 32.1 if secim == "1" else 5.5

            return market_code, market_label, round(inflation, 2), round(wacc, 2)

        print("❌ Geçersiz seçim!")

def tarama_stratejisi_sec():
    print("\n" + "="*75)
    print("🎯 FDS PRO TARAMA STRATEJİSİ".center(75))
    print("="*75)
    print("1️⃣  : Kusursuzlar        👉 [KALİTE + UCUZLUK] (FDS > 7.0 ve Guru Onaylı)")
    print("2️⃣  : Yükselen Yıldızlar 👉 [POTANSİYEL İVME] (FDS > 7.0 ve EPS Büyümesi > %50)")
    print("3️⃣  : Değer Avı          👉 [DERİN İSKONTO] (Graham Onaylı ve İllüzyonsuz)")
    print("4️⃣  : Temiz Bilanço      👉 [MAKSİMUM GÜVENLİK] (Piotroski >= 3 ve Risksiz)")
    print("5️⃣  : Filtresiz          👉 [TÜM PİYASA] (FDS Skoruna Göre Sırala)")
    print("6️⃣  : Fethi Hoca Modeli  👉 [KALİTE + YÜKSEK HIZ] (FDS > 7.0 ve Büyüme > %30)")
    print("7️⃣  : Değer Yaratanlar   👉 [GERÇEK ZENGİNLİK] (ROIC > WACC ve FDS > 6.0)")
    print("-" * 75)
    print("💡 İPUCU: 1. Strateji 'İndirimdeki Mercedes'i, 6. Strateji 'Hızlı Mermi Treni'ni arar.\n")

    while True:
        secim = input("👉 Seçiminiz (1-7) [Varsayılan: 5]: ").strip() or "5"
        isimler = {
            "1": "Kusursuzlar (Kalite + Ucuzluk)",
            "2": "Yükselen Yıldızlar (Potansiyel İvme)",
            "3": "Değer Avı (Derin İskonto)",
            "4": "Temiz Bilanço (Maksimum Güvenlik)",
            "5": "Filtresiz Tüm Piyasa",
            "6": "Fethi Hoca Modeli (Kalite + Yüksek Hız)",
            "7": "Değer Yaratanlar (Gerçek Zenginlik)"
        }
        if secim in isimler:
            return secim, isimler[secim]
        print("❌ Geçersiz seçim!")

# ==================================================================================================
# 🧠 BÖLÜM 2: TRADINGVIEW API VERİ ÇEKME VE FDS PUANLAMA MOTORU
# ==================================================================================================

def get_fundamental_data(market="turkey"):
    """
    TradingView Screener üzerinden tüm hisselerin temel analiz verilerini tek seferde çeker.
    Limit 5000 yapılarak piyasadaki tüm hisselerin eksiksiz gelmesi sağlanır.
    """
    print(f"\n📡 {market.upper()} piyasası için finansal veriler TradingView'dan çekiliyor...")

    fields = [
        'name', 'close', 'volume', 'sector', 'industry',
        'current_ratio', 'debt_to_equity', 'net_debt', 'ebitda',
        'gross_margin', 'operating_margin', 'return_on_equity', 'return_on_invested_capital', 'return_on_assets',
        'total_revenue_yoy_growth_ttm', 'earnings_per_share_diluted_yoy_growth_ttm',
        'free_cash_flow_margin_ttm', 'price_earnings_ttm', 'price_sales_current',
        'enterprise_value_ebitda_ttm', 'price_free_cash_flow_ttm',
        'total_shares_outstanding', 'float_shares_outstanding',
        'market_cap_basic'
    ]

    try:
        # LİMİT EKLENDİ: .limit(5000) ile tüm piyasayı çekiyoruz!
        query = (Query()
                 .set_markets(market)
                 .select(*fields)
                 .limit(20000)
                 .get_scanner_data())

        df = pd.DataFrame(query[1])

        if 'ticker' in df.columns:
            df = df.rename(columns={'ticker': 'Hisse'})
        elif 'name' in df.columns:
            df = df.rename(columns={'name': 'Hisse'})

        print(f"✅ {len(df)} hissenin bilanço verisi başarıyla indirildi.")
        return df

    except Exception as e:
        print(f"❌ Veri çekme hatası: {e}")
        return pd.DataFrame()

def calculate_fds_scores(df, wacc, inflation):
    """
    FDS Pro v3.2 - 5 Ana Grup Puanlama Motoru (Pandas Vektörel Hesaplama)
    Döngü (for) kullanmadan tüm piyasayı aynı anda hesaplar. Yüksek performanslıdır.
    """
    data = df.copy()

    # NaN (Boş) verileri 0 ile doldur (Matematiksel hataları önlemek için)
    data = data.fillna(0)

        # Halka Açıklık Oranı (%) Hesaplama
    data['Halka_Aciklik_Orani'] = np.where(
        data['total_shares_outstanding'] > 0,
        (data['float_shares_outstanding'] / data['total_shares_outstanding']) * 100,
        0
    )

    # ---------------------------------------------------------
    # 💧 G1: LİKİDİTE VE BORÇ YÖNETİMİ (%20 Ağırlık)
    # ---------------------------------------------------------
    # Cari Oran > 1.5 (İyi), Borç/Özkaynak < 50 (İyi)
    data['Score_G1'] = np.where(data['current_ratio'] > 1.5, 10, np.where(data['current_ratio'] > 1.0, 5, 0))
    data['Score_G1'] += np.where(data['debt_to_equity'] < 50, 10, np.where(data['debt_to_equity'] < 100, 5, 0))
    data['Score_G1'] = (data['Score_G1'] / 20) * 100 # 100 üzerinden normalize

    # ---------------------------------------------------------
    # 💰 G2: KÂRLILIK VE VERİMLİLİK (%25 Ağırlık)
    # ---------------------------------------------------------
    # Reel ROE (Enflasyondan arındırılmış) ve ROIC > WACC kontrolü
    data['Real_ROE'] = data['return_on_equity'] - inflation
    data['Score_G2'] = np.where(data['Real_ROE'] > 0, 10, 0)
    data['Score_G2'] += np.where(data['return_on_invested_capital'] > wacc, 10, np.where(data['return_on_invested_capital'] > 0, 5, 0))
    data['Score_G2'] += np.where(data['gross_margin'] > 20, 5, 0)
    data['Score_G2'] = (data['Score_G2'] / 25) * 100

    # ---------------------------------------------------------
    # 📈 G3: BÜYÜME KALİTESİ (%20 Ağırlık)
    # ---------------------------------------------------------
    data['Score_G3'] = np.where(data['total_revenue_yoy_growth_ttm'] > inflation, 10, np.where(data['total_revenue_yoy_growth_ttm'] > 0, 5, 0))
    data['Score_G3'] += np.where(data['earnings_per_share_diluted_yoy_growth_ttm'] > inflation, 10, np.where(data['earnings_per_share_diluted_yoy_growth_ttm'] > 0, 5, 0))
    data['Score_G3'] = (data['Score_G3'] / 20) * 100

    # ---------------------------------------------------------
    # 💎 G4: NAKİT AKIŞ KALİTESİ (%15 Ağırlık)
    # ---------------------------------------------------------
    data['Score_G4'] = np.where(data['free_cash_flow_margin_ttm'] > 10, 15, np.where(data['free_cash_flow_margin_ttm'] > 0, 7.5, 0))
    data['Score_G4'] = (data['Score_G4'] / 15) * 100

    # ---------------------------------------------------------
    # 🏷️ G5: DEĞERLEME (%20 Ağırlık)
    # ---------------------------------------------------------
    # F/K Fallback Mekanizması (F/K anlamsızsa P/S oranına geçiş)
    data['Score_G5'] = np.where((data['price_earnings_ttm'] > 0) & (data['price_earnings_ttm'] < 15), 10,
                       np.where((data['price_earnings_ttm'] <= 0) & (data['price_sales_current'] < 2), 5, 0))
    data['Score_G5'] += np.where((data['enterprise_value_ebitda_ttm'] > 0) & (data['enterprise_value_ebitda_ttm'] < 10), 10, 0)
    data['Score_G5'] = (data['Score_G5'] / 20) * 100

    # ---------------------------------------------------------
    # 🏆 FDS PRO NİHAİ SKOR HESAPLAMASI (Ağırlıklı Ortalama)
    # ---------------------------------------------------------
    data['FDS_Score'] = (
        (data['Score_G1'] * 0.20) +
        (data['Score_G2'] * 0.25) +
        (data['Score_G3'] * 0.20) +
        (data['Score_G4'] * 0.15) +
        (data['Score_G5'] * 0.20)
    )

    # Skoru 10 üzerinden formatla (Örn: 7.5)
    data['FDS_Score'] = (data['FDS_Score'] / 10).round(2)

    # ---------------------------------------------------------
    # 🌟 ÖZKAN FİLİZ / İLKER PARMAKSIZ KRİTERİ (ROIC > WACC)
    # ---------------------------------------------------------
    data['ROIC_WACC_Onayi'] = np.where(
        data['return_on_invested_capital'] > wacc,
        "✅ Geçti",
        "❌ Kaldı"
    )

    return data

# ==================================================================================================
# 🔮 BÖLÜM 3: GURU ANALİZİ VE RİSK RADARI (YAPAY ZEKA KATMANI)
# ==================================================================================================

def calculate_guru_and_risk(df):
    """
    FDS Pro v3.2 - Efsaneler (Guru) Katmanı ve Risk Radarı
    Peter Lynch, Benjamin Graham ve Piotroski mantığını Pandas ile vektörel olarak hesaplar.
    """
    data = df.copy()

    # ---------------------------------------------------------
    # 🧙‍♂️ 1. PETER LYNCH (Büyüme Odaklı Değerleme - PEG Rasyosu)
    # ---------------------------------------------------------
    # PEG = (F/K) / (EPS Büyüme Oranı)
    # Lynch'e göre PEG < 1.0 mükemmel, 1.0 - 1.5 arası makuldür. Negatif büyüme cezalandırılır.
    data['PEG_Ratio'] = np.where(
        data['earnings_per_share_diluted_yoy_growth_ttm'] > 0,
        data['price_earnings_ttm'] / data['earnings_per_share_diluted_yoy_growth_ttm'],
        99.0 # Büyüme yoksa veya negatifse PEG anlamsızdır, yüksek bir değer atanır.
    )

    data['Lynch_Onayi'] = np.where(
        (data['PEG_Ratio'] > 0) & (data['PEG_Ratio'] <= 1.2),
        "✅ Geçti (Ucuz Büyüme)",
        "❌ Kaldı"
    )

    # ---------------------------------------------------------
    # 🏛️ 2. BENJAMIN GRAHAM (Defansif Değer Avcısı)
    # ---------------------------------------------------------
    # Graham'ın defansif kriterleri: Düşük F/K (<15) ve Sağlam Bilanço (Borç/Özkaynak < 50)
    data['Graham_Onayi'] = np.where(
        (data['price_earnings_ttm'] > 0) &
        (data['price_earnings_ttm'] < 15) &
        (data['debt_to_equity'] < 50),
        "✅ İskontolu",
        "❌ Pahalı/Riskli"
    )

    # ---------------------------------------------------------
    # 🌪️ 3. RİSK RADARI: "İFLAS SARMALI" (Altman Z-Score Proxy)
    # ---------------------------------------------------------
    # Aşırı borçlu, özsermaye kârlılığı negatif ve esas faaliyetlerinden zarar eden şirketler.
    data['Risk_Iflas'] = np.where(
        (data['debt_to_equity'] > 150) &
        (data['return_on_equity'] < 0) &
        (data['operating_margin'] < 0),
        "🚨 YÜKSEK RİSK (İflas Sarmalı)",
        "🟢 Güvenli"
    )

    # ---------------------------------------------------------
    # 🔬 4. PIOTROSKI F-SKOR (Finansal Güç ve Muhasebe Testi)
    # ---------------------------------------------------------
    # Tarayıcı (Screener) verileriyle oluşturulmuş 4 Puanlık Hızlı Piotroski Testi
    # 1. ROA Pozitif mi? | 2. Nakit Akışı (FCF) Pozitif mi? | 3. Brüt Marj > %20 mi? | 4. EPS Büyüyor mu?
    p_score = np.zeros(len(data))
    p_score += np.where(data['return_on_assets'] > 0, 1, 0)
    p_score += np.where(data['free_cash_flow_margin_ttm'] > 0, 1, 0)
    p_score += np.where(data['gross_margin'] > 20, 1, 0)
    p_score += np.where(data['earnings_per_share_diluted_yoy_growth_ttm'] > 0, 1, 0)

    data['Piotroski_Proxy'] = p_score # Maksimum 4 puan

    # ---------------------------------------------------------
    # 🎭 5. DEĞER İLLÜZYONU TESPİTİ
    # ---------------------------------------------------------
    # F/K çok düşük (Sanki ucuzmuş gibi) ama şirket aslında küçülüyor (EPS büyümesi negatif).
    data['Risk_Illuzyon'] = np.where(
        (data['price_earnings_ttm'] > 0) &
        (data['price_earnings_ttm'] < 8) &
        (data['earnings_per_share_diluted_yoy_growth_ttm'] < 0),
        "⚠️ Değer İllüzyonu (Tuzak)",
        "Temiz"
    )

    return data

# ==================================================================================================
# 🚀 BÖLÜM 4: ANA TARAMA DÖNGÜSÜ, FİLTRELEME VE EXCEL ÇIKTISI
# ==================================================================================================

def format_fds_sonucu(df):
    # 1. HATA DÜZELTİLDİ: İki liste tek bir doğru listede birleştirildi
    metin_sutunlari = ['Hisse', 'Sektör', 'Piyasa Değeri', 'ROIC > WACC', 'Lynch Onayı', 'Graham Onayı', 'İflas Riski', 'Değer İllüzyonu']
    sayisal_sutunlar = ['Fiyat', 'FDS Skoru', 'Halka Açıklık (%)', 'Piotroski (4)', 'F/K', 'F/K (PEG)', 'ROE (%)', 'ROIC (%)']

    format_dict = {
        'Fiyat': '{:.2f}',
        'FDS Skoru': '{:.2f}',
        'Halka Açıklık (%)': '{:.1f}%',
        'Piotroski (4)': '{:.0f}',
        'F/K': '{:.2f}',
        'F/K (PEG)': '{:.2f}',
        'ROE (%)': '{:.1f}%',
        'ROIC (%)': '{:.1f}%'
    }

    def colorize_skor(val):
        try:
            v = float(val)
            if v >= 7.0: return 'color: white; background-color: #2E8B57; font-weight: bold'
            elif v >= 5.0: return 'color: black; background-color: #FFD700'
            else: return 'color: white; background-color: #DC143C'
        except: return ''

    def colorize_risk(val):
        if 'YÜKSEK RİSK' in str(val) or 'İllüzyonu' in str(val):
            return 'color: white; background-color: #8B0000; font-weight: bold'
        elif 'Güvenli' in str(val) or 'Temiz' in str(val):
            return 'color: green; font-weight: bold'
        return ''

    def colorize_guru(val):
        if '✅' in str(val): return 'color: green; font-weight: bold'
        if '❌' in str(val): return 'color: red'
        return ''

    styled_df = (df.style
                 .set_properties(**{'text-align': 'left'}, subset=metin_sutunlari)
                 .set_properties(**{'text-align': 'center'}, subset=sayisal_sutunlar)
                 .set_table_styles([{'selector': 'th', 'props': [('background-color', '#1E3A8A'), ('color', 'white'), ('text-align', 'center')]}])
                 .applymap(colorize_skor, subset=['FDS Skoru'])
                 .applymap(colorize_risk, subset=['İflas Riski', 'Değer İllüzyonu'])
                 # 2. HATA DÜZELTİLDİ: ROIC > WACC sütunu renklendirme motoruna eklendi
                 .applymap(colorize_guru, subset=['Lynch Onayı', 'Graham Onayı', 'ROIC > WACC']))

    return styled_df.format(format_dict)

def main():
    # 1. Kullanıcı Seçimleri
    market_code, market_label, inflation, wacc = piyasa_secimi()
    strat_code, strat_label = tarama_stratejisi_sec()
    start_ram = psutil.virtual_memory().percent

    print("\n" + "╔" + "═"*78 + "╗")
    print(f"║ 🚀 FDS PRO v3.2 - {market_label.upper()} YAPAY ZEKA TARAMASI BAŞLIYOR ".center(78) + "║")
    print("╚" + "═"*78 + "╝\n")

    print("📊 MAKRO PARAMETRELER:")
    print(f"    ├─ 🌍 Piyasa        : {market_label}")
    print(f"    ├─ 📈 Enflasyon     : %{inflation}")
    print(f"    ├─ 🏦 Risksiz Faiz  : %{wacc} (WACC)")
    print(f"    └─ 🎯 Strateji      : {strat_label}\n")
    print(f"💻 Sistem Durumu: RAM %{start_ram} | CPU %{psutil.cpu_percent()}\n")

    start_time = time.time()

    # 2. Veri Çekme (Bölüm 2)
    raw_data = get_fundamental_data(market=market_code)
    if raw_data.empty:
        print("❌ Veri çekilemedi, program sonlandırılıyor.")
        return

    # 3. Hesaplamalar (Bölüm 2 ve 3)
    print("⚙️ FDS Puanlama Motoru ve Guru Yapay Zekası çalıştırılıyor...")
    scored_data = calculate_fds_scores(raw_data, wacc, inflation)
    final_data = calculate_guru_and_risk(scored_data)

    # 4. Filtreleme (Stratejiye Göre)
    df_filtered = final_data.copy()

    if strat_code == "1": # Kusursuzlar
        df_filtered = df_filtered[(df_filtered['FDS_Score'] >= 7.0) &
                                  ((df_filtered['Lynch_Onayi'].str.contains('✅')) | (df_filtered['Graham_Onayi'].str.contains('✅')))]

    elif strat_code == "2": # Yükselen Yıldızlar (İvme)
        # Not: API anlık veri verdiği için geçmiş bilanço skoru yerine, FDS > 7 ve EPS Büyümesi > %50 olan "İvmeli" hisseleri yakalıyoruz.
        df_filtered = df_filtered[(df_filtered['FDS_Score'] >= 7.0) &
                                  (df_filtered['earnings_per_share_diluted_yoy_growth_ttm'] > 50)]

    elif strat_code == "3": # Değer Avı
        df_filtered = df_filtered[(df_filtered['Graham_Onayi'].str.contains('✅')) &
                                  (df_filtered['Risk_Illuzyon'] == "Temiz")]

    elif strat_code == "4": # Temiz Bilanço
        df_filtered = df_filtered[(df_filtered['Piotroski_Proxy'] >= 3) &
                                  (df_filtered['Risk_Iflas'].str.contains('Güvenli'))]

    elif strat_code == "6": # Fethi Hoca Modeli (İvme ve Kalite)
        df_filtered = df_filtered[(df_filtered['FDS_Score'] >= 7.0) &
                                  (df_filtered['earnings_per_share_diluted_yoy_growth_ttm'] > 30) &
                                  (df_filtered['total_revenue_yoy_growth_ttm'] > 30) &
                                  (df_filtered['Piotroski_Proxy'] >= 3)]

    elif strat_code == "7": # Değer Yaratanlar (Özkan Filiz / İlker Parmaksız Mantığı)
        df_filtered = df_filtered[(df_filtered['ROIC_WACC_Onayi'].str.contains('✅')) &
                                  (df_filtered['FDS_Score'] >= 6.0) &
                                  (df_filtered['Risk_Iflas'].str.contains('Güvenli'))]


    # Piyasa Değerini M, B, T formatına çeviren yardımcı fonksiyon
    def format_market_cap(val):
        try:
            val = float(val)
            if val >= 1_000_000_000_000: return f"{val/1_000_000_000_000:.2f}T"
            elif val >= 1_000_000_000: return f"{val/1_000_000_000:.2f}B"
            elif val >= 1_000_000: return f"{val/1_000_000:.2f}M"
            else: return f"{val:.0f}"
        except:
            return "0"

    # Çeviriciyi Piyasa Değeri sütununa uygula
    df_filtered['market_cap_basic'] = df_filtered['market_cap_basic'].apply(format_market_cap)

    # 5. Sütunları Düzenleme ve Temizleme (Excel ve Ekran için)
    display_columns = {
        'Hisse': 'Hisse',
        'close': 'Fiyat', # YENİ
        'market_cap_basic': 'Piyasa Değeri', # YENİ
        'sector': 'Sektör',
        'FDS_Score': 'FDS Skoru',
        'Halka_Aciklik_Orani': 'Halka Açıklık (%)',
        'Piotroski_Proxy': 'Piotroski (4)',
        'price_earnings_ttm': 'F/K',
        'PEG_Ratio': 'F/K (PEG)',
        'return_on_equity': 'ROE (%)',
        'return_on_invested_capital': 'ROIC (%)',
        'ROIC_WACC_Onayi': 'ROIC > WACC',
        'Lynch_Onayi': 'Lynch Onayı',
        'Graham_Onayi': 'Graham Onayı',
        'Risk_Iflas': 'İflas Riski',
        'Risk_Illuzyon': 'Değer İllüzyonu'
    }

    # Sadece istediğimiz sütunları al ve yeniden adlandır
    df_final = df_filtered[list(display_columns.keys())].rename(columns=display_columns)

    # FDS Skoruna göre yüksekten düşüğe sırala
    df_final = df_final.sort_values(by='FDS Skoru', ascending=False).reset_index(drop=True)

    # 6. Sonuç Raporu
    end_time = time.time()
    toplam_sure = end_time - start_time
    end_ram = psutil.virtual_memory().percent

    print("\n" + "="*80)
    print("                              📊 TARAMA TAMAMLANDI                               ")
    print("="*80)
    print(f"⏱️  Toplam Süre: {toplam_sure:.2f} saniye")
    print(f"✅ Sinyal Bulunan: {len(df_final)} Hisse")
    print(f"💻 RAM: %{start_ram} → %{end_ram}")
    print("="*80 + "\n")

    if not df_final.empty:
        print("📋 DETAYLI İSTATİSTİK SONUÇLARI:")

        # Hisse adından Borsa ismini ayıkla (Örn: "NASDAQ:AAPL" -> "NASDAQ")
        df_final['Borsa'] = df_final['Hisse'].apply(lambda x: x.split(':')[0] if ':' in str(x) else 'BIST')

        filename = f"FDS_PRO_SCREENER_{market_code.upper()}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

        try:
            if market_code == "america":
                # ABD için Excel'de çoklu sekme (writer) oluştur
                with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                    borsalar = df_final['Borsa'].unique()

                    for borsa in borsalar:
                        # Sadece o borsaya ait hisseleri filtrele
                        df_borsa = df_final[df_final['Borsa'] == borsa].copy()

                        # Ekrana ayrı tablo olarak bas
                        print(f"\n🏛️ {borsa} BORSASI SONUÇLARI ({len(df_borsa)} Hisse)")
                        display(format_fds_sonucu(df_borsa.drop(columns=['Borsa']).head(50)))

                        # Excel'e ayrı sekme (sheet) olarak kaydet
                        df_borsa.drop(columns=['Borsa']).to_excel(writer, sheet_name=str(borsa)[:31], index=False)

                print(f"\n💾 Sonuçlar başarıyla Excel'e kaydedildi: {filename}")
                print("💡 Not: Excel dosyasında NASDAQ, NYSE, AMEX ve OTC ayrı sekmeler (sheet) olarak ayrıldı!")

            else:
                # BIST için tek tablo (Klasik görünüm)
                display(format_fds_sonucu(df_final.drop(columns=['Borsa']).head(50)))
                df_final.drop(columns=['Borsa']).to_excel(filename, index=False)
                print(f"\n💾 Sonuçlar başarıyla Excel'e kaydedildi: {filename}")

        except Exception as e:
            print(f"\n❌ Excel'e kaydetme hatası: {e}")
    else:
        print("\n⚠️ Belirlediğiniz kriterlere uygun hisse bulunamadı. Piyasa şartları zorlu olabilir!")

if __name__ == "__main__":
    main()




📦 Gerekli kütüphaneler kuruluyor/güncelleniyor...
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.4 MB/s eta 0:00:00
✅ Kurulum tamamlandı.




                   🌍 HEDEF PİYASA SEÇİMİ                    
1️⃣  : Borsa İstanbul (BIST) - [Canlı Makro Veri]
2️⃣  : ABD Borsaları (NASDAQ/NYSE) - [Canlı Makro Veri]

👉 Seçiminiz (1-2) [Varsayılan: 1]: 1

📡 Borsa İstanbul için güncel makro veriler TradingView'dan çekiliyor...
✅ Canlı Veri Alındı -> Enflasyon: %32.11 | WACC: %31.96

                        🎯 FDS PRO TARAMA STRATEJİSİ                        
1️⃣  : Kusursuzlar        👉 [KALİTE + UCUZLUK] (FDS > 7.0 ve Guru Onaylı)
2️⃣  : Yükselen Yıldızlar 👉 [POTANSİYEL İVME] (FDS > 7.0 ve EPS Büyümesi > %50)
3️⃣  : Değer Avı          👉 [DERİN İSKONTO] (Graham Onaylı ve İllüzyonsuz)
4️⃣  : Temiz Bilanço      👉 [MAKSİMUM GÜVENLİK] (Piotroski >= 3 ve Risksiz)
5️⃣  : Filtresiz          👉 [TÜM PİYASA] (FDS Skoruna Göre Sırala)
6️⃣  : Fethi Hoca Modeli  👉 [KALİTE + YÜKSEK HIZ] (FDS > 7.0 ve Büyüme > %30)
7️⃣  : Değer Yaratanlar   👉 [GERÇEK ZENGİNLİK] (ROIC > WACC ve FDS > 6.0)
----------------------------------------------------------------

,Hisse,Fiyat,Piyasa Değeri,Sektör,FDS Skoru,Halka Açıklık (%),Piotroski (4),F/K,F/K (PEG),ROE (%),ROIC (%),ROIC > WACC,Lynch Onayı,Graham Onayı,İflas Riski,Değer İllüzyonu
0,BIST:TERA,159.70,117.60B,Finance,9.50,31.4%,3,2.38,0.01,132.9%,132.8%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
1,BIST:GEDIK,6.51,10.75B,Finance,9.50,49.2%,4,4.64,0.01,39.2%,38.7%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
2,BIST:ATATP,236.20,5.44B,Technology Services,9.50,1.6%,4,3.03,0.00,64.1%,63.5%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
3,BIST:RGYAS,203.00,67.86B,Finance,8.50,24.6%,4,3.66,0.00,14.8%,12.5%,❌ Kaldı,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
4,BIST:DUNYH,145.00,5.25B,Consumer Non-Durables,8.50,0.0%,3,6.63,0.04,63.3%,63.3%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
5,BIST:RAYSG,170.60,28.70B,Finance,8.50,5.0%,3,7.25,0.09,52.8%,52.7%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
6,BIST:PAGYO,155.00,13.89B,Finance,8.50,64.8%,4,8.10,0.06,11.6%,11.6%,❌ Kaldı,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
7,BIST:AKGRT,7.05,11.64B,Finance,8.50,28.0%,3,4.73,0.05,36.3%,35.4%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
8,BIST:ISGSY,18.47,9.47B,Finance,8.50,41.0%,3,1.97,0.01,53.9%,48.9%,✅ Geçti,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz
9,BIST:GLYHO,17.22,33.84B,Finance,8.50,63.9%,4,6.35,0.12,40.8%,7.9%,❌ Kaldı,✅ Geçti (Ucuz Büyüme),✅ İskontolu,🟢 Güvenli,Temiz



💾 Sonuçlar başarıyla Excel'e kaydedildi: FDS_PRO_SCREENER_TURKEY_20260719_1605.xlsx
